## Basic Statistics-2  Assignment_4

## Problem Statement : Hospital Patient Data Analysis
> Context:
- A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

### TASKS: 

#### 1.	Load the patient dataset and show summary with info().

In [196]:
import pandas as pd
import numpy as np
from scipy import stats

In [197]:
#Load Dataset in Patient
patient_data=pd.read_csv('Patient_Data.csv')
#Display a result
patient_data

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,10-01-2023 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,11-01-2023 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,12-01-2023 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,13-01-2023 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,14-01-2023 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,10-01-2023 09:00


In [198]:
#Show Summary
patient_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


#### 2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [199]:
billing_columns_data = patient_data[['PatientID', 'Department', 'Doctor', 'BillAmount']]

billing_columns_data.head()

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


#### 3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [200]:
#Drop columns
patient_df = patient_data.drop(columns=['ReceptionistID', 'CheckInTime'])

patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


#### 4.	Use groupby to find total bill amount per department.

In [201]:
# Group by Department and calculate total bill amount
total_bill_dept = patient_data.groupby('Department')['BillAmount'].sum()

# Display result
total_bill_dept

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64

#### 5.	Remove duplicate patient records based on PatientID.

In [202]:
# Remove duplicate patients (keep first occurrence)
patient_data = patient_data.drop_duplicates(subset='PatientID', keep='first')

patient_data.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,10-01-2023 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,11-01-2023 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,12-01-2023 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,13-01-2023 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,14-01-2023 08:45


#### 6.	Fill missing BillAmount values with the mean bill amount

In [203]:
# Make sure it's a proper copy
patient_data = patient_data.copy()

# Calculate mean
mean_bill = patient_data['BillAmount'].mean()

# Fill missing values properly
patient_data['BillAmount'] = patient_data['BillAmount'].fillna(mean_bill)

# Round properly
patient_data['BillAmount'] = patient_data['BillAmount'].round(0).astype(int)

patient_data

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000,1,10-01-2023 09:00
1,102,Bob,Neurology,Dr. John,6233,2,11-01-2023 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500,1,12-01-2023 11:00
3,104,David,Cardiology,Dr. Smith,6200,3,13-01-2023 12:00
4,105,Eva,Dermatology,Dr. Rose,6233,2,14-01-2023 08:45


#### 7.	Merge the billing dataset with patient dataset on PatientID

In [204]:
# Load billing dataset
bill_data = pd.read_csv("billing_data.csv")
bill_data

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [205]:
# Merge both datasets
merged_df = pd.merge(patient_data, bill_data, on='PatientID', how='left')

merged_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000,1,10-01-2023 09:00,2000,3000
1,102,Bob,Neurology,Dr. John,6233,2,11-01-2023 10:30,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500,1,12-01-2023 11:00,2500,5000
3,104,David,Cardiology,Dr. Smith,6200,3,13-01-2023 12:00,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233,2,14-01-2023 08:45,1000,4000


#### 8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [206]:
new_patients_df = pd.DataFrame({
    'PatientID': [106, 107],
    'Name': ['Rahul', 'Sneha'],
    'Department': ['Neurology', 'Cardiology'],
    'Doctor': ['Dr. John', 'Dr. Smith'],
    'BillAmount': [4800, 6100],
    'ReceptionistID': [2, 1],
    'CheckInTime': ['15-01-2023 09:30', '15-01-2023 11:00']
})

In [207]:
# Row-wise concatenation
updated_patient_df = pd.concat([patient_data, new_patients_df],
                               axis=0,
                               ignore_index=True)

updated_patient_df.tail()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
2,103,Charlie,Orthopedics,Dr. Lee,7500,1,12-01-2023 11:00
3,104,David,Cardiology,Dr. Smith,6200,3,13-01-2023 12:00
4,105,Eva,Dermatology,Dr. Rose,6233,2,14-01-2023 08:45
5,106,Rahul,Neurology,Dr. John,4800,2,15-01-2023 09:30
6,107,Sneha,Cardiology,Dr. Smith,6100,1,15-01-2023 11:00


#### 9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [208]:
# Concatenate column-wise
final_df = pd.concat([patient_data, bill_data[['InsuranceCovered', 'FinalAmount']]], axis=1)

final_df.head(6)

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000,1,10-01-2023 09:00,2000,3000
1,102,Bob,Neurology,Dr. John,6233,2,11-01-2023 10:30,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500,1,12-01-2023 11:00,2500,5000
3,104,David,Cardiology,Dr. Smith,6200,3,13-01-2023 12:00,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233,2,14-01-2023 08:45,1000,4000


## Expected Outcome:
>  -	Final cleaned dataset with accurate billing info.
>  -	All missing values handled, merged dataset across PatientID.
>  -	Ability to perform further analytics on department-wise revenue or doctor performance.
